# hexbot · Advanced internals

What the framework is actually doing under the hood: the C game engine, MCTS math, the policy/value heads, the training loss, and the optimisations that let the loop run on a laptop. Each section has runnable cells that show the numbers, not just describe them.

Prerequisite: you have worked through the [guided tour](guided_tour.ipynb).

**Table of contents**

1. The C game engine: bitboards, candidates, undo
2. Board encoding: what the network actually sees
3. The three network heads: policy, value, threat
4. MCTS math: UCB and PUCT
5. Decision walkthrough: a single move, traced end to end
6. Training loss
7. Optimisations that matter
8. Forced moves and threat search: cheap deterministic wins
9. Where to look in the source

In [ ]:
!pip install --quiet hexbot

## 1. The C game engine: bitboards, candidates, undo

The hot path in this framework is `place / undo / scored_moves`. Pure Python would manage maybe 30k operations per second. The C engine in `engine.c` hits **1.4 million** place/undo cycles per second on M-series silicon by doing two things:

1. **Bitboard win detection.** Each player's stones live in a packed 19x19 bitfield. A win check is a handful of bit shifts and population counts. No per-cell scanning.
2. **Incremental candidate sets.** The board is infinite in principle, so enumerating every legal cell is impossible. Instead the engine maintains a set of cells within a few rings of any existing stone. `legal_moves()` returns that set. `place(q, r)` adds neighbours of `(q, r)`; `undo()` removes them. The C engine stores this on a stack with no allocation.

You can measure the throughput yourself:

In [ ]:
from hexbot import HexGame
import time

g = HexGame()
g.place(0, 0); g.place(1, 0); g.place(1, -1)

N = 200_000
t0 = time.perf_counter()
for _ in range(N):
    g.place(2, 0)
    g.undo()
dt = time.perf_counter() - t0
print(f"place/undo throughput: {N/dt/1e6:.2f} M/sec ({dt:.2f}s for {N:,} cycles)")

## 2. Board encoding: what the network actually sees

The network takes a fixed-size tensor `(7, 19, 19)`. That is seven 19x19 channels:

| Channel | Meaning |
|---:|---|
| 0 | Current player's stones (binary) |
| 1 | Opponent's stones (binary) |
| 2 | Empty cells (binary) |
| 3 | Indicator: player to move (all 0 or all 1) |
| 4 | Move-count-in-turn (1/2 stones into the current turn) |
| 5 | Threat map: cells where current player has a 4-in-a-row latent line |
| 6 | Threat map: same for opponent |

The board is infinite, so the framework centres the 19x19 window on the centroid of existing stones and gives back the offset `(oq, or_)` so you can map model output coordinates back to game coordinates.

In [ ]:
from hexbot import encode_state

g = HexGame()
g.place(0, 0)
g.place(2, 0); g.place(2, -1)

tensor, oq, orr = encode_state(g)
print(f"shape:    {tuple(tensor.shape)}")
print(f"offsets:  (oq={oq}, or_={orr})")
print(f"player-0 stones channel sum: {int(tensor[0].sum())}")
print(f"player-1 stones channel sum: {int(tensor[1].sum())}")
print(f"empty cells channel sum:     {int(tensor[2].sum())}")

## 3. The three network heads: policy, value, threat

The default Orca network is a ResNet-style trunk (12 residual blocks, 128 filters) feeding **three** output heads:

$$\pi(s) = \text{softmax}(W_\pi h(s)) \in \mathbb{R}^{19 \times 19}$$

The policy head $\pi$ is a probability distribution over the 361 candidate cells. We sometimes call this the prior because MCTS uses it to bias exploration.

$$v(s) = \tanh(W_v h(s)) \in [-1, +1]$$

The value head $v$ is a scalar: $+1$ if the side to move is winning, $-1$ if losing, $0$ for an even position.

$$\tau(s) \in \mathbb{R}^{4}$$

The threat head $\tau$ is an auxiliary task: predict the count of latent 4-in-a-row lines for both players. It does not affect inference but it stabilises training.

Let us actually evaluate the network on a position:

In [ ]:
from hexbot import nn_evaluate
import numpy as np

result = nn_evaluate(g)
print(f"value: {result['value']:+.3f}  (network's estimate; positive = good for side to move)")

policy = result['policy']            # dict {(q, r): prob}
top = sorted(policy.items(), key=lambda kv: -kv[1])[:5]
print("\ntop 5 policy entries (raw, no MCTS):")
for move, p in top:
    bar = '#' * int(p * 80)
    print(f"  {move}  p={p:.3f}  {bar}")

## 4. MCTS math: UCB and PUCT

Monte Carlo Tree Search builds a tree of game states by repeatedly:

1. **Select** a path from the root by following the action with the highest UCB score until a leaf is reached.
2. **Expand + evaluate** the leaf: query the network for the policy prior $\pi$ and value $v$.
3. **Backup** $v$ along the visited path, incrementing each node's visit count and updating its mean Q.

Vanilla UCB1 (used by classical Go bots before AlphaZero) picks the action $a$ that maximises

$$Q(s, a) + c \sqrt{\frac{\ln N(s)}{N(s, a)}}$$

where $N(s)$ is the parent's visit count, $N(s, a)$ is the child's visit count, $Q(s,a)$ is the child's mean value. The square-root term shrinks as the child gets visited more often, so the bonus encourages exploring less-visited children.

AlphaZero replaces UCB1 with **PUCT** (Predictor + UCT):

$$Q(s, a) + c_\text{puct} \cdot \pi(s, a) \cdot \frac{\sqrt{N(s)}}{1 + N(s, a)}$$

The key change is multiplying the exploration term by the network's prior $\pi(s, a)$. Moves the network thinks are bad get less exploration; moves it likes get more. This is what lets MCTS converge orders of magnitude faster than uniform-policy MCTS.

`c_puct` controls the explore/exploit balance. Orca uses `C_PUCT = 1.5` (in `orca/config.py`). Higher = more exploration, lower = more exploitation.

In [ ]:
# Compare: raw policy vs MCTS visit distribution after 50, 200, 800 sims.
# Watch which moves get amplified by search.
from hexbot import mcts_search

def show_top(label, items, total):
    print(f"\n{label}")
    for move, count in items[:5]:
        frac = count / total
        bar = '#' * int(frac * 60)
        print(f"  {move}  {frac*100:5.1f}%  {bar}")

raw = sorted(policy.items(), key=lambda kv: -kv[1])
show_top("raw policy prior (no search)", raw, sum(policy.values()))

for sims in [50, 200, 800]:
    r = mcts_search(g, sims=sims)
    total = sum(v for _, v in r['top_moves'])
    show_top(f"MCTS visit distribution (sims={sims})", r['top_moves'], total)

Notice how visit counts converge as `sims` grows. The top move usually surfaces by 50 sims already; the long tail (sims 200 to 800) is mostly about re-distributing visits among the second and third candidates.

## 5. Decision walkthrough: a single move, traced end to end

Now let us walk through exactly what Orca does to pick a move in a sharp position.

In [ ]:
from hexbot import find_forced_move, find_threats, nn_evaluate, mcts_search

# A position with mild tension
g = HexGame()
g.place(0, 0)
g.place(2, 0); g.place(2, -1)
g.place(1, 0); g.place(0, 1)
g.place(3, 0); g.place(3, -1)

print("Step 1: forced moves?")
forced = find_forced_move(g)
print(f"  find_forced_move -> {forced}")
if forced is None:
    print("  no forced move; continue")

In [ ]:
print("Step 2: threats on the board")
threats = find_threats(g)
print(f"  {len(threats)} threat cells: {threats[:5]}{'...' if len(threats) > 5 else ''}")

In [ ]:
print("Step 3: raw network evaluation")
ev = nn_evaluate(g)
print(f"  value: {ev['value']:+.3f}  (side-to-move's outlook)")
top_policy = sorted(ev['policy'].items(), key=lambda kv: -kv[1])[:3]
print("  top 3 policy priors:")
for m, p in top_policy:
    print(f"    {m}  p={p:.3f}")

In [ ]:
print("Step 4: MCTS-amplified decision")
r = mcts_search(g, sims=200)
print(f"  MCTS picks: {r['best_move']}")
print("  top 3 visit counts:")
for m, n in r['top_moves'][:3]:
    print(f"    {m}  visits={n}")

print("\nFinal decision: Orca plays", r['best_move'])

Compare the raw-policy top with the MCTS top. Often they agree; when they disagree, it is a sign that the prior was wrong about a move that does not hold up under deeper search. This is exactly what AlphaZero training corrects over iterations: the visit distribution becomes the new policy target.

## 6. Training loss

Three terms, summed:

$$\mathcal{L} = \underbrace{-\sum_a \hat{\pi}(s, a) \log \pi_\theta(s, a)}_{\text{policy: cross-entropy}} \;+\; \underbrace{(z - v_\theta(s))^2}_{\text{value: MSE}} \;+\; \lambda \cdot \mathcal{L}_\text{threat}(s)$$

Where:

- $\hat{\pi}$ is the **MCTS visit distribution** at $s$ during self-play (the better-than-the-network policy that came out of search).
- $z \in \{-1, 0, +1\}$ is the eventual game outcome from $s$'s perspective.
- $v_\theta$ and $\pi_\theta$ are the network's value and policy heads.
- $\mathcal{L}_\text{threat}$ is an auxiliary MSE on the 4-in-a-row count for both players.

The first two terms are pure AlphaZero. The threat term is hexbot-specific and improves stability early in training when the policy and value heads have not learned much yet.

**Why this loop converges.** Each iteration produces self-play data where moves were chosen by `argmax(visits) ≈ argmax(MCTS-amplified policy)`, which is stronger than `argmax(network policy alone)`. Training pushes $\pi_\theta$ toward that distribution, so the next iteration's network is stronger than the last. Self-play data quality grows with the network. It is a positive feedback loop bounded only by network capacity and time.

## 7. Optimisations that matter

### a. C engine (already covered)

Bitboard win detection + incremental candidate set + zero-allocation undo. The 50x speedup vs Python is what makes the rest possible.

### b. Hex symmetry augmentation (8x data per game)

A hex board has rotational symmetry around any point. Every training sample $(s, \pi, z)$ can be transformed by 3 grid-safe rotations + 4 axial rotations to produce up to 7 additional samples, all with the same outcome. The framework filters out rotations that push stones off the 19x19 window; typical yield is 5-7x more data per actual game played.

In [ ]:
from hexbot import augment_sample
from orca.data import TrainingSample
import torch, numpy as np

# Build a fake training sample
s = TrainingSample(
    encoded_state=torch.randn(7, 19, 19),
    policy_target=np.random.dirichlet(np.ones(361)),
    player=0, result=1.0,
)
augs = augment_sample(s)
print(f"original sample -> {len(augs)} samples after augmentation")
print(f"each transformed copy is a valid training sample")

### c. Mixed precision (FP16, CUDA only)

On NVIDIA tensor cores, FP16 matmul is ~2x faster than FP32 with negligible accuracy loss. The trainer uses PyTorch's `GradScaler` to keep gradients well-scaled. Enable with the default training pipeline on CUDA; the trainer auto-detects and turns it off on MPS where FP16 paths are flaky.

### d. ONNX export to worker subprocesses

Self-play happens in process pool workers (one per CPU core). If each worker loaded a fresh PyTorch model, network setup would dominate runtime. Instead the trainer exports the current network to ONNX once per iteration and the workers load the ONNX file. ONNX Runtime inference is several times faster than PyTorch on CPU and the export is amortised across many games.

### e. Batched MCTS

A naive MCTS evaluates one position per network forward pass. `BatchedMCTS` collects up to 64 leaves before issuing a single GPU forward. Throughput on a GPU grows almost linearly with batch size up to the tensor core limit.

### f. C MCTS (GIL-free, threaded)

The tree search itself is implemented in C with explicit thread support, so multiple GPU-bound threads can build trees concurrently without fighting the Python GIL. See `orca/c_mcts.py`. The Python `BatchedMCTS` remains the default; `C MCTS` shines for high-throughput training on multi-GPU CUDA hosts.

### g. Replay buffer with priority sampling

Not all samples are equal. The buffer assigns priority based on:

- Game length (very short games are dropped, long games get bonus).
- Tactical value (samples around forced-move resolutions get a 3.0x boost).
- TD error (samples where the network's value was very wrong get amplified).

Training samples a minibatch proportional to priority, so the optimiser spends its budget on the highest-information samples.

## 8. Forced moves and threat search: cheap deterministic wins

Before invoking MCTS, the bot checks two cheap functions:

- `find_forced_move`: if the opponent threatens an immediate win, return the only move that blocks it. Skips MCTS entirely.
- `threat_search(game, depth)`: short-depth tactical search that walks combinations of threats to find forced wins several moves out.

These are the equivalent of "tactical search" in chess engines. They're cheap, deterministic, and catch positions where MCTS would otherwise waste many sims rediscovering the obvious answer.

In [ ]:
# Confirm the forced-move shortcut on a five-in-a-row threat
from hexbot import find_forced_move

g = HexGame()
g.place(10, 10)
g.place(0, 0); g.place(1, 0)   # P1 starts a horizontal line
g.place(11, 11); g.place(12, 12)
g.place(2, 0); g.place(3, 0)   # P1 four in a row at (0..3, 0)
g.place(13, 13); g.place(14, 14)
g.place(4, 0)                  # P1 plays first stone of next turn: now five! (-1,0) or (5,0) wins

print(f"P1 winning moves: {list(__import__('hexbot').find_winning_moves(g, player=1))}")
print(f"forced move for P1 (their second stone this turn): {find_forced_move(g)}")

## 9. Where to look in the source

Pointers for when you want to read the actual implementation:

| Topic | File |
|---|---|
| C game engine: bitboards, win detection, candidates | `engine.c` |
| Python `HexGame` API wrapping the C engine via ctypes | `hexgame.py` |
| Network architectures (8 variants) | `orca/network.py`, `orca/hex_conv.py`, `orca/hex_gnn.py`, etc. |
| PUCT/MCTS in Python (batched, NN-guided) | `orca/search.py` |
| C MCTS (GIL-free, threaded) | `orca/c_mcts.py`, `engine.c` |
| Training loop | `orca/train.py` |
| Loss functions (policy CE + value MSE + threat aux) | `bot.py` (`train_step`) |
| Replay buffer with priority sampling | `orca/replay.py` |
| Hex symmetry augmentation | `orca/augment.py` |
| ONNX export for workers | `bot.py` (`export_onnx`) |
| Endgame solver (alpha-beta + TT cache) | `orca/solver.py` |
| Opening book (trie from winning games) | `orca/openings.py` |

## What to do with this knowledge

- Use `--config large` once you understand the loss math: more capacity helps once the data quality is high.
- Tune `C_PUCT` (in `orca/config.py`) for your hardware: lower exploration is faster on slow hardware where you can't afford to play out alternatives.
- Add a custom auxiliary head if you have a domain signal that should be learned (the framework's plugin system lets you register a custom network).
- Read the [Training Guide](https://github.com/Saiki77/hexbot-building-framework/wiki/Training-Guide) wiki page for the operator's view of the same machinery.

## Deeper dives

This notebook is the overview. Two focused companion notebooks go further:

- [advanced_mcts_and_training.ipynb](advanced_mcts_and_training.ipynb): Dirichlet root noise, temperature scheduling, threat-map / C-heuristic blending into the prior, all eight architectures, cosine annealing, GradScaler, gradient clipping, soft vs one-hot policy targets, temporal value decay, replay buffer priority math, progressive game-length filters, tactical priority boosts, AutoTuner rules, curriculum learning, plateau detection.
- [advanced_engine_and_search.ipynb](advanced_engine_and_search.ipynb): bitboard win detection, incremental candidate set, Zobrist hashing, alpha-beta with transposition table, killer moves, late move reduction, iterative deepening, threat search algorithm, endgame solver, opening book, ONNX export pipeline, GPU inference server, C MCTS.